# Multiva — run the whole pipeline on a free GPU

Dub a talking-head video into an Indian language — transcribe, translate, clone
the speaker's voice, lip-sync — on the machine this notebook is attached to,
not on your laptop.

**Before you start:** this needs a **GPU runtime**.
In Colab: **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.

> First run takes 10-20 minutes: the install pulls PyTorch and the ML stack.
> Later runs in the same session are fast because pip caches the wheels.
>
> On a GPU the 7-second sample clip renders in well under a minute; on a CPU the
> same clip takes several minutes. That difference is the point of this notebook:
> the engine already prefers CUDA when it sees it, so the only thing to fix is
> giving it a GPU.


## 0. Get the code

Colab gives a GPU VM with a writable `/content`. We clone the repository there and install FFmpeg, which every render shells out to.

In [ ]:
import os
import shutil
import subprocess

REPO_URL = "https://github.com/gajjalaashok75-UI/video-dubbing"
REPO_DIR = "/content/video-dubbing"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning the repository...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository already present; updating...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"],
                   check=False)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=False)

if not shutil.which("ffmpeg"):
    print("Installing FFmpeg...")
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)

print("FFmpeg:", shutil.which("ffmpeg"))
print("Ready: ", REPO_DIR)

## 1. Check the machine

Confirms the GPU is visible, reports how much VRAM it has, and picks the quality preset. Nothing here downloads models - that happens in step 4 - so this cell is quick.

In [ ]:
import os
import sys

REPO_DIR = "/content/video-dubbing"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from engine import deploy_helpers as runtime

print("Platform:", runtime.detect_platform())
print("Python:  ", sys.version.split()[0])

report = runtime.hardware_report()
if report["cuda_available"]:
    print(f"GPU:      {report['gpu_name']} ({report['vram_gb']} GB VRAM)")
else:
    print("GPU:      none - renders run on CPU and are slow")
print("Device:  ", report["device"])
print("Preset:  ", report["recommended_preset"])
for note in report["notes"]:
    print("  note:", note)

print("FFmpeg:  ", "ready" if runtime.ensure_ffmpeg() else "NOT available")

## 2. Point the runtime at a writable directory

Uploads, finished renders and the model cache are written at run time. This cell puts them under the runtime root instead of inside the checkout, which is small and not durable.

**It must run before the server is imported**, because the engine reads the upload directory at import time.

In [ ]:
dirs = runtime.configure_runtime()

print("Runtime root:", next(iter(dirs.values())).parent)
for name, path in dirs.items():
    print(f"  {name:8} -> {path}")

# Finished videos land here; remembered for later cells.
OUTPUT_DIR = dirs["output"]

## 3. Install the application

Two installs: the CUDA PyTorch wheels first, then the rest of the pinned dependencies. This is the slow cell - expect 5-10 minutes on the first run.

Installing PyTorch on its own from the `cu124` index is what makes the GPU get used. Installing the plain `requirements.txt` alone would fetch the CPU-only wheels and every render would silently fall back to the CPU.

In [ ]:
import subprocess
import sys

print("Installing the CUDA PyTorch build (the slow part)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "torch>=2.2.0", "torchaudio>=2.2.0", "torchvision>=0.17.0",
     "--index-url", "https://download.pytorch.org/whl/cu124"],
    check=True,
)

cmd = [sys.executable, "-m", "pip", "install", "-r", f"{REPO_DIR}/requirements.txt"]
print("Installing:", " ".join(cmd))
print("Python:", sys.version.split()[0])

# Stream pip to the notebook so a failure shows which package died, not just
# that something did.
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()

if proc.returncode != 0:
    print()
    print("=" * 60)
    print("INSTALL FAILED. The real cause is in the pip output above.")
    print("=" * 60)
    raise SystemExit("The install failed; see the pip output above this line.")

print("\nInstall complete.")

## 4. Fetch the models

About 7.5 GB, resumable, and only once per runtime. Without this the first render stalls for several minutes fetching weights mid-job.

One model is **gated**: `ai4bharat/IndicF5`, the voice that speaks the 11 Indian languages. It needs a Hugging Face token (see the token cell). Everything else downloads either way; without the token those languages are simply unavailable.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/download_models.py"], check=False)

## 5. Start the server

Starts the API **and** the built studio interface (the repo ships `web/`, so no Node build is needed) in a background thread of this kernel. The cell prints the public URL to open.

Re-running the cell stops the previous server first, so you never end up with two processes fighting over the port.

In [ ]:
# Stop any server a previous run of this cell left behind.
_previous = globals().get("server")
if _previous is not None:
    _previous.stop()
    print("Stopped the previous server.")

PORT = 8000
server = runtime.start_server(host="0.0.0.0", port=PORT, ready_timeout=600)

url = runtime.notebook_public_url(PORT)
print()
print("=" * 62)
print("  Multiva is running")
print(f"  Open:  {url}/app/")
print("=" * 62)
print()
print("Keep this notebook tab open while you use the app.")

# On Colab this URL is proxied automatically and needs no account or token.
# It is valid only while this notebook is running.

## Use it

1. Open the URL printed above (append `/app/`).
2. On the **Dub** page, drop in a talking-head video and pick the source and
   target languages.
3. Press **Start** and watch the progress. Renders are written to the output
   directory from step 2.

### Optional: a Hugging Face token

The voice model for the Indian languages is gated. Set a token before starting
the server to unlock it; without one those languages report as unavailable and
the pipeline falls back to a default voice.

Accept the licence at <https://huggingface.co/ai4bharat/IndicF5>, create a read
token at <https://huggingface.co/settings/tokens>, then run the cell below and
re-run the model and server cells.

In [ ]:
import os

# os.environ["HF_TOKEN"] = "hf_..."   # or set it in the platform's secrets panel

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is set. Re-run the 'Fetch the models' cell, then restart the server.")
else:
    print("No HF_TOKEN set. Indian-language voice cloning will be unavailable;")
    print("translation and subtitles work in every language either way.")

## 7. Stop the server

Frees the port and the GPU memory. Run this when you are done, or just close the notebook - the server is a daemon thread and dies with the kernel.

In [ ]:
server.stop()
print("Server stopped:", not server.is_running())

## Troubleshooting

| Symptom | Fix |
|---|---|
| `The server did not become ready ...` | Re-run the install cell; a partial install is the usual cause. Scroll up in the server cell for the traceback. |
| `ModuleNotFoundError: No module named 'torch'` | The CUDA install cell did not finish. Re-run it and read its output. |
| `GPU: none — renders run on CPU` | The runtime is not a GPU one. Change it (see the top) and re-run from cell 1. |
| `401`/`403` when fetching models | The gated voice model needs a Hugging Face token. Accept the licence at <https://huggingface.co/ai4bharat/IndicF5>, then set `HF_TOKEN` in the token cell and re-run the model + server cells. |
| Renders are still slow | Confirm the GPU is visible in cell 1 (the `GPU:` line). A CPU render is correct, just slow. |
| Out of memory on a small GPU | Use a smaller Whisper: `os.environ["WHISPER_MODEL"] = "medium"` before starting the server. |
